<div style="background:#123B63;color:white;padding:14px 18px;border-radius:6px">
<b>CSE 816 &mdash; Machine Learning Lab</b> &nbsp;&middot;&nbsp; Department of CSE, University of Chittagong<br>
<span style="font-size:90%">Module 1 &middot; Week 2 &middot; Part 3 of 4 &nbsp;&middot;&nbsp; 60 minutes</span>
</div>

# Feature Scaling and the Learning Rate

Part 2 ended with a correct algorithm that was useless: 50,000 iterations and still
crawling, with any $\\alpha$ above $0.0018$ blowing up. Nothing was wrong with the code. This
lab fixes it in three lines, then builds a systematic procedure for choosing $\\alpha$ so you
never have to guess again.

**Companion theory lecture:** CSE 815, Week 2, Part 2 (Making Gradient Descent Work in Practice).

## What you will be able to do

1. Explain why unequal feature ranges force $\alpha$ to be tiny, and measure it.
2. Implement the three scaling methods from the lecture and compare their output ranges.
3. Fit $\mu$ and $\sigma$ on training data only, and apply them everywhere else.
4. Run the $\times 3$ search for $\alpha$ and read the resulting learning curves.
5. Convert scaled weights back to original units so they can be interpreted.

---

## 1. Where we left off

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

%matplotlib inline
try:
    plt.style.use("seaborn-v0_8-whitegrid")
except OSError:
    plt.style.use("ggplot")

np.set_printoptions(precision=4, suppress=True)

FEATURES = ["size (k sq ft)", "bedrooms", "floor", "age (yr)"]


def load_flats(seed=815, m=60):
    """Chattogram flats: four features, price in lakh BDT."""
    rng = np.random.default_rng(seed)
    size  = np.round(rng.uniform(0.60, 3.00, m), 2)
    beds  = np.clip(np.round(1.2 + 1.7 * size + rng.normal(0, 0.6, m)), 1, 6)
    floor = rng.integers(1, 13, m).astype(float)
    age   = np.round(rng.uniform(1, 60, m), 0)
    price = (22.0 * size + 2.5 * beds + 0.9 * floor - 0.35 * age + 8.0
             + rng.normal(0, 2.5, m))
    return np.column_stack([size, beds, floor, age]), np.round(price, 1)


def predict(X, w, b):
    return X @ w + b


def compute_cost(X, y, w, b):
    err = predict(X, w, b) - y
    return float(np.sum(err ** 2) / (2 * X.shape[0]))


def compute_gradient(X, y, w, b):
    m = X.shape[0]
    err = predict(X, w, b) - y
    return X.T @ err / m, float(np.sum(err) / m)


def gradient_descent(X, y, alpha, num_iters, w_init=None, b_init=0.0, record_every=1):
    """Batch gradient descent; carried over unchanged from Part 2."""
    w = np.zeros(X.shape[1]) if w_init is None else np.array(w_init, dtype=float)
    b = float(b_init)
    history = {"cost": []}
    for i in range(num_iters):
        dj_dw, dj_db = compute_gradient(X, y, w, b)
        w = w - alpha * dj_dw
        b = b - alpha * dj_db
        if i % record_every == 0 or i == num_iters - 1:
            history["cost"].append(compute_cost(X, y, w, b))
    return w, b, history


X_train, y_train = load_flats()
m, n = X_train.shape
print("X_train.shape:", X_train.shape)

In [ ]:
# The exact minimum, for reference throughout this lab.
A = np.column_stack([X_train, np.ones(m)])
coef, *_ = np.linalg.lstsq(A, y_train, rcond=None)
w_exact, b_exact = coef[:n], coef[n]
J_min = compute_cost(X_train, y_train, w_exact, b_exact)

print("closed-form minimum:")
print("  w =", w_exact)
print("  b = %.4f" % b_exact)
print("  J = %.6f" % J_min)

---

## 2. Diagnosing the problem

The lecture's argument: a feature with a **large** range needs a **small** weight, and vice
versa. So $J$ is hypersensitive to some parameters and nearly flat in others, and the cost
contours become long, thin and tilted.

That vague picture has a precise measurement. Gradient descent on a quadratic cost is stable
only when

$$\alpha < \alpha_{\max} = \frac{2}{\lambda_{\max}},$$

where $\lambda_{\max}$ is the largest eigenvalue of the curvature matrix
$H = \frac{1}{m}X^{\top}X$ (with the column of ones included). The ratio
$\kappa = \lambda_{\max}/\lambda_{\min}$ &mdash; the **condition number** &mdash; is how
elongated the valley is, and it sets how many iterations you need.

In [ ]:
def curvature(X):
    """Largest/smallest eigenvalue of (1/m) A^T A, and the resulting stability limit on alpha."""
    A = np.column_stack([X, np.ones(X.shape[0])])
    H = A.T @ A / X.shape[0]
    ev = np.linalg.eigvalsh(H)
    return ev.max(), ev.min(), 2.0 / ev.max(), ev.max() / ev.min()


lam_max, lam_min, a_max, kappa = curvature(X_train)
print("raw features")
print(f"  lambda_max = {lam_max:.4e}")
print(f"  lambda_min = {lam_min:.4e}")
print(f"  alpha_max  = {a_max:.6f}    <- anything above this diverges")
print(f"  kappa      = {kappa:.4e}    <- how elongated the valley is")

`alpha_max = 0.0018` is exactly the wall you hit in Part 2. And $\kappa \approx 1.8\times10^4$
says the valley is roughly $\sqrt{\kappa} \approx 135$ times longer than it is wide.

### Seeing the valley

Two parameters at a time, holding the rest at their optimum. Compare a well-matched pair with
a badly matched one.

In [ ]:
def cost_slice(j1, j2, X, y, w_star, b_star, span=2.5, pts=90):
    """J over a grid in (w_j1, w_j2), with all other parameters held at their optimum."""
    s1 = span * max(abs(w_star[j1]), 1e-3)
    s2 = span * max(abs(w_star[j2]), 1e-3)
    g1 = np.linspace(w_star[j1] - s1, w_star[j1] + s1, pts)
    g2 = np.linspace(w_star[j2] - s2, w_star[j2] + s2, pts)
    G1, G2 = np.meshgrid(g1, g2)
    Z = np.empty_like(G1)
    w = w_star.copy()
    for r in range(pts):
        for c in range(pts):
            w[j1], w[j2] = G1[r, c], G2[r, c]
            Z[r, c] = compute_cost(X, y, w, b_star)
    return G1, G2, Z


fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.2))
for ax, (j1, j2) in zip(axes, [(0, 1), (0, 3)]):
    G1, G2, Z = cost_slice(j1, j2, X_train, y_train, w_exact, b_exact)
    lv = np.logspace(np.log10(Z.min() + 1e-6), np.log10(Z.max()), 22)
    ax.contour(G1, G2, Z, levels=lv, cmap="viridis", alpha=0.8)
    ax.scatter([w_exact[j1]], [w_exact[j2]], c="#C97B17", s=140, marker="*", zorder=5)
    ax.set_xlabel(f"w for {FEATURES[j1]}")
    ax.set_ylabel(f"w for {FEATURES[j2]}")
    ax.set_title(f"{FEATURES[j1]}  vs  {FEATURES[j2]}")
plt.tight_layout()
plt.show()

The right-hand panel is the problem in one picture. Size and age differ in range by a factor
of about twenty, and the contours degenerate into a narrow diagonal trough. Steepest descent
is perpendicular to the local contour, so from most starting points it points **across** the
trough rather than along it: the path zig-zags and $\alpha$ must stay tiny to avoid
overshooting the walls.

---

## 3. Three ways to scale

From the lecture, applied to feature $j$ with mean $\mu_j$, standard deviation $\sigma_j$:

| Method | Formula | Typical output |
|---|---|---|
| divide by the max | $x_j / \max_j$ | $(0, 1]$, not centred |
| mean normalization | $(x_j - \mu_j)/(\max_j - \min_j)$ | roughly $[-0.5, 0.5]$ |
| z-score | $(x_j - \mu_j)/\sigma_j$ | mean 0, std 1 |

In [ ]:
def scale_max(X):
    return X / X.max(axis=0)


def scale_mean_norm(X):
    return (X - X.mean(axis=0)) / (X.max(axis=0) - X.min(axis=0))


def scale_zscore(X):
    """Z-score normalization. Returns the scaled matrix AND the statistics used."""
    mu = X.mean(axis=0)            # axis=0 -> one number per FEATURE
    sigma = X.std(axis=0)
    return (X - mu) / sigma, mu, sigma


X_z, mu, sigma = scale_zscore(X_train)

print("mu    =", mu)
print("sigma =", sigma)
print()
print(f"{'feature':16s} {'raw range':>22} {'/max':>16} {'mean-norm':>18} {'z-score':>18}")
print("-" * 94)
for j in range(n):
    r  = X_train[:, j]; a = scale_max(X_train)[:, j]
    bb = scale_mean_norm(X_train)[:, j]; c = X_z[:, j]
    print(f"{FEATURES[j]:16s} [{r.min():8.2f},{r.max():8.2f}] "
          f"[{a.min():5.2f},{a.max():5.2f}] [{bb.min():6.2f},{bb.max():6.2f}] "
          f"[{c.min():6.2f},{c.max():6.2f}]")

In [ ]:
# Verify the defining property of the z-score: every column has mean 0 and std 1.
print("column means:", X_z.mean(axis=0).round(12))
print("column stds :", X_z.std(axis=0).round(12))

assert np.allclose(X_z.mean(axis=0), 0.0, atol=1e-12)
assert np.allclose(X_z.std(axis=0), 1.0, atol=1e-12)
print("\nz-scoring verified.")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11.5, 3.8))
for ax, (M, title) in zip(axes, [(X_train, "Raw features"), (X_z, "After z-scoring")]):
    # The keyword for tick labels was renamed in matplotlib 3.9; set them afterwards instead.
    ax.boxplot([M[:, j] for j in range(n)], widths=0.55)
    ax.set_xticklabels(["size", "beds", "floor", "age"])
    ax.set_title(title)
    ax.set_ylabel("value")
plt.tight_layout()
plt.show()

The left plot is unreadable for the first three features because age dominates the vertical
axis &mdash; which is the whole complaint. On the right every feature occupies roughly the
same interval, comfortably inside the lecture's $-1 \le x_j \le 1$ rule of thumb (age reaches
$1.93$, which is fine; the rule is an order of magnitude, not a bound).

### What scaling did to the curvature

In [ ]:
lam_max_z, lam_min_z, a_max_z, kappa_z = curvature(X_z)

print(f"{'':12s} {'alpha_max':>12} {'kappa':>14}")
print("-" * 40)
print(f"{'raw':12s} {a_max:12.6f} {kappa:14.4e}")
print(f"{'z-scored':12s} {a_max_z:12.6f} {kappa_z:14.4e}")
print(f"\nalpha_max is {a_max_z / a_max:,.0f} times larger.")
print(f"kappa fell by a factor of {kappa / kappa_z:,.0f}.")

---

## 4. The payoff

Same data, same code, same starting point. Only the feature matrix changed.

In [ ]:
def iters_to_reach(X, y, alpha, target, cap=200_000):
    """Number of iterations for gradient descent from the origin to reach J <= target."""
    w = np.zeros(X.shape[1]); b = 0.0
    for i in range(1, cap + 1):
        dj_dw, dj_db = compute_gradient(X, y, w, b)
        w = w - alpha * dj_dw
        b = b - alpha * dj_db
        if compute_cost(X, y, w, b) <= target:
            return i
    return None


target = 1.01 * J_min
print(f"target: J <= 1.01 * J_min = {target:.6f}\n")

n_raw = iters_to_reach(X_train, y_train, 0.0015, target)
n_scaled = iters_to_reach(X_z, y_train, 0.5, target)

print(f"  raw features,    alpha = 0.0015 : {n_raw:,} iterations")
print(f"  z-scored,        alpha = 0.5    : {n_scaled:,} iterations")
print(f"\n  {n_raw / n_scaled:,.0f} times fewer.")

In [ ]:
w_z, b_z, hist_z = gradient_descent(X_z, y_train, alpha=0.5, num_iters=200)
_, _, hist_raw = gradient_descent(X_train, y_train, alpha=0.0015, num_iters=200)

plt.figure(figsize=(7, 4.2))
plt.plot(hist_raw["cost"], c="#9B2C4B", lw=2, label="raw features, alpha = 0.0015")
plt.plot(hist_z["cost"], c="#1B7F79", lw=2, label="z-scored, alpha = 0.5")
plt.axhline(J_min, ls="--", c="#C97B17", lw=1.4, label=f"minimum J = {J_min:.3f}")
plt.yscale("log")
plt.xlabel("iteration")
plt.ylabel("J(w, b)   (log scale)")
plt.title("The first 200 iterations")
plt.legend(fontsize=8)
plt.show()

print(f"after 200 iterations:  raw J = {hist_raw['cost'][-1]:10.4f}"
      f"    scaled J = {hist_z['cost'][-1]:.6f}   (minimum {J_min:.6f})")

---

## 5. Choosing $\alpha$

The lecture's procedure: try values spaced by roughly $\times 3$, run each for a small fixed
number of iterations, and pick the largest $\alpha$ whose curve still decreases smoothly.

In [ ]:
import warnings

alphas = [0.001, 0.003, 0.01, 0.03, 0.1, 0.3, 1.0, 3.0]
runs = {}

with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    for a in alphas:
        _, _, h = gradient_descent(X_z, y_train, alpha=a, num_iters=200)
        runs[a] = np.array(h["cost"])

def is_healthy(costs):
    """A run is healthy when every recorded cost is finite and never increases."""
    return bool(np.isfinite(costs).all() and np.all(np.diff(costs) <= 1e-12))


print(f"{'alpha':>8} {'J after 200':>16} {'verdict':>14}")
print("-" * 41)
for a, c in runs.items():
    ok = is_healthy(c)
    print(f"{a:>8g} {c[-1]:16.4g} {('decreasing' if ok else 'DIVERGING'):>14}")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))

for a, c in runs.items():
    if is_healthy(c):
        ax[0].plot(c, lw=1.8, label=f"alpha = {a:g}")
ax[0].axhline(J_min, ls="--", c="#C97B17", lw=1.2, label=f"minimum J = {J_min:.3f}")
ax[0].set_yscale("log"); ax[0].set_xlabel("iteration"); ax[0].set_ylabel("J   (log scale)")
ax[0].set_title("The runs that decreased")
ax[0].legend(fontsize=7)

# The diverging run needs its own axes: after 200 iterations it reaches about 1e275,
# which no shared axis can display alongside the others.
ax[1].plot(runs[3.0][:10], c="#9B2C4B", lw=2, marker="o", ms=4)
ax[1].set_yscale("log"); ax[1].set_xlabel("iteration"); ax[1].set_ylabel("J   (log scale)")
ax[1].set_title("alpha = 3.0: J rises on every step")

plt.tight_layout()
plt.show()

print(f"alpha = 3.0 after 200 iterations:  J = {runs[3.0][-1]:.4g}")

The rule of thumb picks the largest $\alpha$ that still decreases smoothly, then backs off one
step. Here everything up to $1.0$ decreases and $3.0$ explodes, so $0.3$ is the sensible
choice &mdash; and the eigenvalue calculation tells us exactly where the wall is.

Note that the diverging run never produced `inf` or `nan` within 200 iterations: it simply
grew, to about $10^{275}$. A test based on `np.isfinite` alone would have passed it. **The
test that matters is whether $J$ ever increased.**

In [ ]:
print(f"predicted stability limit  alpha_max = {a_max_z:.6f}")

with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    for a in [1.00, 1.02, 1.03, 1.05]:
        _, _, h = gradient_descent(X_z, y_train, alpha=a, num_iters=300)
        c = np.array(h["cost"])
        verdict = "converges" if np.isfinite(c[-1]) and c[-1] < 10 else "DIVERGES"
        print(f"  alpha = {a:.2f}  ->  {verdict}")

The changeover happens exactly where $2/\lambda_{\max}$ said it would. This is worth knowing
even though you will not compute eigenvalues in practice: the $\times 3$ search is not
folklore, it is a cheap way to find a number that is genuinely determined by the data.

> **If $J$ rises with a very small $\alpha$** &mdash; say $10^{-6}$ &mdash; the bug is in the
> gradient, not the learning rate. A correct gradient with a tiny step *must* decrease $J$.

---

## 6. Fit on train, apply everywhere

$\mu_j$ and $\sigma_j$ are **parameters learned from the training set**. Compute them there,
store them, and reuse those same numbers for every other data set. Recomputing them elsewhere
leaks information and silently corrupts your results.

In [ ]:
rng = np.random.default_rng(0)
idx = rng.permutation(m)
train_idx, test_idx = idx[:45], idx[45:]

X_tr, y_tr = X_train[train_idx], y_train[train_idx]
X_te, y_te = X_train[test_idx], y_train[test_idx]
print(f"train: {X_tr.shape[0]} examples    held out: {X_te.shape[0]} examples")

# Fit the scaler on TRAINING data only.
X_tr_s, mu_tr, sigma_tr = scale_zscore(X_tr)
print("\nmu    (train only) =", mu_tr)
print("sigma (train only) =", sigma_tr)

# Apply the STORED statistics to the held-out data. Do not refit.
X_te_s = (X_te - mu_tr) / sigma_tr

w_tr, b_tr, _ = gradient_descent(X_tr_s, y_tr, alpha=0.5, num_iters=500)
print(f"\n  J on training data : {compute_cost(X_tr_s, y_tr, w_tr, b_tr):.4f}")
print(f"  J on held-out data : {compute_cost(X_te_s, y_te, w_tr, b_tr):.4f}")

In [ ]:
# Now do it the wrong way: rescale the held-out set using its OWN mean and std.
X_te_wrong, _, _ = scale_zscore(X_te)

J_right = compute_cost(X_te_s, y_te, w_tr, b_tr)
J_wrong = compute_cost(X_te_wrong, y_te, w_tr, b_tr)

print(f"held-out J, stored mu/sigma (correct) : {J_right:8.4f}")
print(f"held-out J, refitted mu/sigma (wrong) : {J_wrong:8.4f}")
print(f"\nThe bug makes the model look {J_wrong / J_right:.0f} times worse than it is.")

assert J_wrong > 5 * J_right

Note which direction the damage runs here. Refitting the scaler shifted the held-out flats
onto a different scale from the one the weights were trained for, so the reported cost is
*wrong*, not merely optimistic. In other settings the same mistake flatters the model
instead. Either way the number you report is not measuring what you think.

> **In practice:** wrap $\mu$ and $\sigma$ with the model and save them together. A model
> without its scaler is not a model. Part 4 shows how scikit-learn packages this as a
> `Pipeline`.

---

## 7. Scaled weights mean something different

After scaling, $w_j$ is the effect of a **one-standard-deviation** change in feature $j$, not
of one square foot or one year. Predictions are unaffected; interpretations are not.

In [ ]:
w_z, b_z, _ = gradient_descent(X_z, y_train, alpha=0.5, num_iters=500)

print(f"{'feature':16s} {'scaled w_j':>12} {'sigma_j':>10} {'raw w_j':>12}")
print("-" * 54)
w_unscaled = w_z / sigma
b_unscaled = b_z - np.sum(w_z * mu / sigma)
for j in range(n):
    print(f"{FEATURES[j]:16s} {w_z[j]:12.4f} {sigma[j]:10.4f} {w_unscaled[j]:12.4f}")
print(f"{'bias b':16s} {b_z:12.4f} {'':10s} {b_unscaled:12.4f}")

print("\nclosed-form solution on raw features:")
print("  w =", w_exact, " b = %.4f" % b_exact)
assert np.allclose(w_unscaled, w_exact, atol=1e-2)
assert np.isclose(b_unscaled, b_exact, atol=1e-2)
print("\nUnscaling recovers the raw-feature solution exactly.")

Read the two weight columns side by side. On the **raw** scale age has the smallest weight of
the four ($-0.37$ lakh per year) and looks unimportant. On the **scaled** scale it has the
largest magnitude ($-6.06$ lakh per standard deviation) and is the second strongest driver of
price after size &mdash; because age varies over 57 years while size varies over 2.4 thousand
square feet.

> Neither column is "the truth". The raw weight answers *what does one more year cost?*; the
> scaled weight answers *which feature moves the price most across the range this data
> actually covers?* Quote the one that answers the question you were asked, and say which it
> is.

### Checkpoint 1

The two scaled features that matter most are size and age. Redraw the contour plot from
section 2 for that pair, but using `X_z` and the scaled optimum `w_z`, `b_z`. Compare it with
the raw version.

*Expected:* contours that are close to circular rather than a narrow diagonal trough.

In [ ]:
# Your code here

### Checkpoint 2

Write `fit_scaled(X, y, alpha, num_iters)` that z-scores `X`, runs gradient descent, and
returns a dictionary with keys `"w"`, `"b"`, `"mu"`, `"sigma"` &mdash; everything needed to
make a prediction later. Then write `predict_new(model, x_raw)` that takes one raw,
**unscaled** example and returns the predicted price.

Test it on the flat from Part 2: size `1.80`, `3` bedrooms, floor `5`, age `12` years.

*Expected:* approximately `56.21` lakh BDT &mdash; the same answer as the closed-form model in
Part 2, because scaling changes the path to the minimum, not the minimum itself.

In [ ]:
# Your code here

---

## 8. Recap

- Unequal feature ranges make the cost contours a long thin trough. This is measurable:
  $\alpha_{\max} = 2/\lambda_{\max}$ and $\kappa = \lambda_{\max}/\lambda_{\min}$.
- On this data set scaling raised $\alpha_{\max}$ from `0.0018` to `1.03` and cut the
  iterations needed by a factor of several hundred.
- `mu = X.mean(axis=0)`, `sigma = X.std(axis=0)` &mdash; `axis=0`, always.
- Fit $\mu$ and $\sigma$ on the training set, store them, and apply the stored values
  everywhere else.
- Search $\alpha$ over $\times 3$ steps; take the largest that decreases smoothly, then back
  off one.
- If $J$ rises with a tiny $\alpha$, the gradient is wrong, not $\alpha$.
- Scaled weights are per standard deviation. Unscale with $w_j^{\text{raw}} = w_j/\sigma_j$
  and $b^{\text{raw}} = b - \sum_j w_j\mu_j/\sigma_j$.

### Exercises to hand in

1. Repeat the `alpha_max` and `kappa` calculation for mean normalization and for
   divide-by-max. Rank the three methods, and say whether the ranking justifies z-scoring
   being the default.
2. Multiply the age column by 1000 (record it in days rather than years) and refit **without**
   scaling. Report the new `alpha_max`, and the iterations needed to reach `1.01 * J_min`.
   Then scale and refit. Explain why a change of units, which carries no new information, can
   change the running time by orders of magnitude.
3. Implement `scale_zscore` for a feature that is **constant** across every example. What
   happens, and why? Propose a fix and justify the value you substitute.
4. Automate the lecture's procedure: write `search_alpha(X, y, alphas, num_iters)` returning
   the largest $\alpha$ whose cost history is finite and monotonically decreasing, backed off
   by one grid step. Report what it chooses for the raw and the scaled data.
5. Repeat the train/held-out experiment in section 6 for ten different random splits. Report
   the mean and standard deviation of the held-out cost, and comment on whether a single split
   was enough to judge the model.

### Next

**Part 4 &mdash; Feature Engineering, Polynomial Regression and scikit-learn:** build new
features from old ones, fit curves with a linear model, and check your work against a library.

**Reading:** James et al., *ISL* 2e, &sect;3.3.